[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-glm.ipynb)

# Generalized Linear Models & Poisson Regression

*AIBits Academy · Machine Learning End To End · Supervised Learning · New*

Linear and Logistic Regression are not two unrelated algorithms — they're two special cases of one unifying framework, which also covers a third case essential for count data.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **🎯 Intuition First**
>
> Linear and logistic regression feel like two separate tools — one predicts a number, the other a probability. GLMs reveal they're the **same machine with a different output adapter**. Keep the linear engine η = Xθ untouched; just swap the "link function" that connects it to what you're predicting — identity for a plain number, logit for a probability, log for a count. Same core, three jobs.

## The Unifying Framework

A Generalized Linear Model has three components: a **linear predictor** η = Xθ (identical across every GLM), a **link function** g that connects η to the mean of y, and a **distribution** for y from the exponential family.

$$g(E[y\mid x]) = \eta = \mathbf{x}^{\top}\theta \quad\Longleftrightarrow\quad E[y\mid x] = g^{-1}(\mathbf{x}^{\top}\theta)$$

| Model | Distribution of y | Link function g | Use for |
|---|---|---|---|
| Linear Regression | Gaussian | Identity: g(μ)=μ | Continuous, unbounded (price, temperature) |
| Logistic Regression | Bernoulli | Logit: g(μ)=ln(μ/(1−μ)) | Binary outcome (default / no default) |
| **Poisson Regression** | Poisson | Log: g(μ)=ln(μ) | Non-negative integer counts (calls, defects, visits) |

This is exactly why the gradient you derived on the Logistic Regression page had the identical form ∇J = XᵀT(ŷ−y) as Linear Regression's — every GLM fit by maximum likelihood produces this same gradient shape, with only the definition of ŷ = g⁻¹(Xθ) changing.

## Try It — One η, Three Link Functions

Drag the linear predictor η and watch the *same* value flow through all three link functions at once — this is the "one unifying framework" claim made concrete.

## Poisson Regression — Modelling Counts

Linear regression assumes y is continuous and can be negative — nonsensical for a count like "number of customer complaints this week" (always a non-negative integer). Poisson regression instead assumes y | x follows a Poisson distribution with rate λ = E[y|x]:

$$P(y=k\mid\lambda) = \frac{\lambda^{k}e^{-\lambda}}{k!} \qquad \log(\lambda) = \mathbf{x}^{\top}\theta \quad\Longrightarrow\quad \lambda = e^{\mathbf{x}^{\top}\theta}$$

The log-link guarantees λ > 0 for any θ — exactly the constraint a count must satisfy. Coefficients are interpreted **multiplicatively**: a one-unit increase in xⱼ multiplies the expected count by e^θⱼ, not adds to it.

## Code — Call Centre Volume Prediction (Paytm Customer Support)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Daily support call volume: [is_payday, app_outage_hrs, promo_campaign]
np.random.seed(3)
n = 300
is_payday   = np.random.choice([0,1], n, p=[0.93,0.07])
outage_hrs  = np.random.exponential(0.3, n).clip(0,6)
promo       = np.random.choice([0,1], n, p=[0.8,0.2])
log_lambda  = 4.2 + 0.9*is_payday + 1.1*outage_hrs + 0.4*promo
calls = np.random.poisson(np.exp(log_lambda / 5))  # scaled for realistic daily volume

df = pd.DataFrame({'calls':calls, 'is_payday':is_payday,
                    'outage_hrs':outage_hrs, 'promo':promo})

model = smf.glm('calls ~ is_payday + outage_hrs + promo', data=df,
                family=sm.families.Poisson()).fit()
print(model.summary().tables[1])
# exp(coef) = multiplicative effect on expected call volume
print("\nMultiplicative effects:")
print(np.exp(model.params))

## Checking the Poisson Assumption — Overdispersion

Poisson regression assumes Var(y) = E[y] (mean equals variance). Real count data is often **overdispersed** (variance > mean) — e.g., a small number of "viral" support-ticket days blow out the variance far beyond what Poisson predicts.

> **⚠ When Overdispersion Appears**
>
> Check: if `calls.var() / calls.mean()` is well above 1, standard errors from plain Poisson regression are too small (falsely confident p-values). Fix: use a **Negative Binomial** regression instead (`sm.families.NegativeBinomial()`) — it adds an extra dispersion parameter to absorb the excess variance without changing the log-link interpretation.

## GLM Family Cheat Sheet

| Data type | GLM family | Indian business example |
|---|---|---|
| Continuous, symmetric | Gaussian (Linear Regression) | Apartment price in ₹ lakhs |
| Binary | Bernoulli (Logistic Regression) | Loan default yes/no |
| Counts (equidispersed) | Poisson | Daily support calls, defect counts per batch |
| Counts (overdispersed) | Negative Binomial | Viral-prone ticket volume, insurance claims |
| Proportions/rates | Binomial (grouped logistic) | Click-through rate per ad impression batch |

> **🔗 Real-World Link — Instagram Reach Analysis**
>
> Impressions on a post are count data, not a continuous measurement — exactly the case Poisson regression is built for. A real 119-post dataset shows impressions correlating strongly with new follows (r=0.889). [See the case study →](https://statso.io/instagram-reach-analysis-case-study/) ·

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Fit a Poisson regression

Support-call counts follow a Poisson law whose rate depends on `promo_days`. Fit `sm.GLM(y, sm.add_constant(x), family=sm.families.Poisson())` and store the fitted parameters (intercept, slope) in `params`.

In [ ]:
import numpy as np, statsmodels.api as sm
rng = np.random.default_rng(1)
x = rng.uniform(0, 2, 600)
y = rng.poisson(np.exp(0.5 + 0.8 * x))
params = None   # TODO


In [ ]:
try:
    check("intercept near 0.5", abs(params[0] - 0.5) < 0.15)
    check("slope near 0.8", abs(params[1] - 0.8) < 0.15)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np, statsmodels.api as sm
rng = np.random.default_rng(1)
x = rng.uniform(0, 2, 600)
y = rng.poisson(np.exp(0.5 + 0.8 * x))
params = sm.GLM(y, sm.add_constant(x), family=sm.families.Poisson()).fit().params

```

</details>

### Exercise 2 · Medium · Interpret the coefficient as a rate ratio

In a log-link model, `exp(coef)` is a multiplicative effect. Store in `rate_ratio` the factor by which the expected call count changes for **one extra promo day**, and `pct_change` = `(rate_ratio - 1) * 100`.

In [ ]:
import numpy as np
rate_ratio = pct_change = None   # TODO (use params[1] from the previous exercise)


In [ ]:
try:
    check("rate ratio about e^0.8 = 2.2", 1.9 < rate_ratio < 2.6)
    check("percent change", abs(pct_change - (rate_ratio - 1) * 100) < 1e-9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
rate_ratio = float(np.exp(params[1]))
pct_change = (rate_ratio - 1) * 100

```

</details>

### Exercise 3 · Stretch · Detect overdispersion

The counts below vary more than a Poisson allows. Fit a Poisson GLM and compute the dispersion `disp = result.pearson_chi2 / result.df_resid` (about 1 if Poisson is right). Store it in `disp` and set `overdispersed` to `disp > 1.5`.

In [ ]:
import numpy as np, statsmodels.api as sm
rng = np.random.default_rng(2)
x = rng.uniform(0, 2, 600)
mu = np.exp(0.5 + 0.8 * x)
y = rng.negative_binomial(2, 2 / (2 + mu))
disp = overdispersed = None   # TODO


In [ ]:
try:
    check("dispersion well above 1", disp > 1.5)
    check("flag", overdispersed is True)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np, statsmodels.api as sm
rng = np.random.default_rng(2)
x = rng.uniform(0, 2, 600)
mu = np.exp(0.5 + 0.8 * x)
y = rng.negative_binomial(2, 2 / (2 + mu))
res = sm.GLM(y, sm.add_constant(x), family=sm.families.Poisson()).fit()
disp = res.pearson_chi2 / res.df_resid
overdispersed = bool(disp > 1.5)

```

When dispersion is far above 1 use a negative-binomial (or quasi-Poisson) model, otherwise the standard errors are too optimistic.

</details>

---
*Back to the course: **Machine Learning End To End → Generalized Linear Models & Poisson Regression**.*